In [13]:
from spin_lattices import (
    KagomeLattice,
    SpinLattice,
    ChainLattice,
    SquareLattice,
    TriangleLattice,
)
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from pathlib import Path
import networkx as nx
import numpy as np
from typing import Callable
import torch
import numpy.typing as npt
import lattice_symmetries as ls
from typing import Any, Optional, Union, Dict, Tuple
from loguru import logger
from collections import namedtuple
from torch import Tensor
import torch.nn as nn
from misc_utils import make_unpacked_configurations
from vmc_vs_lbfgs_2023_08_02 import LogProbDenseNet
import io
from contextlib import redirect_stderr
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from nqs_playground_helpers import (
    SamplingOptions,
    split_into_batches,
    safe_exp,
    sample_exactly,
    sample_full,
    forward_with_batches,
)
from scipy.sparse import csr_matrix, coo_matrix, diags
from scipy.sparse.csgraph import connected_components
import sys
from kagome_cnn import KagomeCNNRegression
from torch.nn.utils import parameters_to_vector
import time

from my_stopwatch import stopwatch, Stopwatch
from misc_utils import torch_overlap as find_overlap
from vmc_amplitude import LogProbDenseNetPairwiseXor
import itertools
import fire
from misc_utils import differentiable_safe_exp
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from misc_utils import torch_overlap as overlap

# TensorDataset
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import Adam

In [6]:
lattice = KagomeLattice(2, 4)
system = HeisenbergJ1J2(
    lattice,
    1,
    1,
    use_symmetries=False,
    spin_inversion=None,
    ground_state_cache_dir=Path("groundstates"),
)
system.get_eigenstates(1)

[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


(array([-43.03958899, -42.82459918, -42.7457909 , -42.64435572,
        -42.638814  , -42.638814  , -42.56683915, -42.56683915,
        -42.56683915, -42.34783072]),
 array([[ 4.90411613e-09,  6.52025278e-08,  3.40441081e-08, ...,
          3.10850235e-09,  1.62077234e-08,  1.75996851e-08],
        [ 2.80245214e-08,  1.91775319e-08,  1.66782353e-10, ...,
          1.19297566e-09,  6.99751282e-08, -8.55468716e-08],
        [ 4.11419221e-19,  1.86579185e-18, -1.17569822e-08, ...,
         -1.36109137e-08,  2.42753447e-08, -4.70584705e-19],
        ...,
        [-2.25499309e-20, -1.76496527e-18, -1.17569822e-08, ...,
         -1.36109137e-08,  2.42753447e-08,  4.84731975e-19],
        [ 2.80245213e-08,  1.91775319e-08,  1.66782350e-10, ...,
          1.19297566e-09,  6.99751282e-08, -8.55468716e-08],
        [ 4.90411612e-09,  6.52025278e-08,  3.40441081e-08, ...,
          3.10850235e-09,  1.62077234e-08,  1.75996851e-08]]))

In [20]:
n_train = 1000
n_test = 10000
samples = np.random.choice(
    system.canonical_basis.states,
    size=n_train + n_test,
    replace=True,
    p=system.get_ground_state_in_canonical_basis() ** 2,
)
samples_train = samples[:n_train]
samples_test = samples[n_train:]

log_probs_train = torch.from_numpy(
    np.log(np.abs(system.get_ground_state_coeffs(samples_train))).astype(np.float32) * 2
)
log_probs_test = torch.from_numpy(
    np.log(np.abs(system.get_ground_state_coeffs(samples_test))).astype(np.float32) * 2
)

In [34]:
def n_params(xor_pairs, n_hidden):
    model = LogProbDenseNetPairwiseXor(system, n_hidden=n_hidden, xor_pairs=xor_pairs)
    return sum(p.nelement() for p in model.parameters())

In [46]:
all_pairwise = tuple(
    map(
        np.array,
        zip(*itertools.combinations(range(system.number_spins), 2)),
    )
)

In [47]:
n_params(xor_pairs=all_pairwise, n_hidden=512)

154625

In [48]:
n_params(xor_pairs=([], []), n_hidden=512)

13313

In [59]:
min(
    [
        (
            abs(
                n_params(
                    xor_pairs=all_pairwise,
                    n_hidden=n_hidden,
                )
                - n_params(xor_pairs=([], []), n_hidden=8192)
            ),
            n_hidden,
        )
        for n_hidden in range(600, 800)
    ]
)

(82, 705)

In [54]:
len(all_pairwise[0])

276

In [61]:
lr = 1e-3

# xor_pairs = tuple(
#     map(np.array, zip(*itertools.combinations(range(system.number_spins), 2)))
# )

# xor_pairs = ([], [])
xor_pairs = all_pairwise
model = LogProbDenseNetPairwiseXor(system, n_hidden=8192, xor_pairs=xor_pairs)

# Create a TensorDataset from your inputs X and Y
dataset = TensorDataset(
    torch.from_numpy(samples_train.astype(np.float32)), log_probs_train
)

# Create a DataLoader for your dataset with a batch size of 32 and shuffling
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Define a loss function - Mean Squared Error (MSE) for regression
criterion = torch.nn.MSELoss()

# Define an optimizer - Adam
optimizer = Adam(model.parameters(), lr=lr)  # Learning rate

# Number of epochs (iterations over the entire dataset)
epochs = 500

# Create a SummaryWriter instance for TensorBoard
# writer = SummaryWriter(
#     log_dir=(
#         f"experiments/2023_07_03/{datetime.now().strftime('%H_%M_%S')}"
#         f"ch=_{','.join(map(str, channels))}_{','.join(additional_generators)}_{lr=}_{filter_sizes=}"
#     )
# )

writer = SummaryWriter(
    log_dir=(
        f"experiments/{datetime.now().strftime('%Y_%m_%d')}_dense_xor_supervised/{datetime.now().strftime('%H_%M_%S')}"
        #        f"xor={xor_masks_idxs}_ch=_{channels}_{lr=}_{blocks=}"
    )
)
for epoch in range(epochs):
    running_loss = 0.0
    for action_index, data in enumerate(dataloader, 0):
        # Get the inputs and move them to the specified device
        inputs, log_amplitudes = data

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # weights = torch.exp(2 * log_amplitudes)
        # weights = weights / torch.sum(weights)

        # Compute loss
        loss = (
            # weights @
            criterion(outputs, log_amplitudes.view(-1, 1))
        )  # Reshape labels to match output shape

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Collect loss
        running_loss += loss.item()

    # Add average loss per epoch to TensorBoard
    writer.add_scalar("Training Loss", running_loss / len(dataloader), epoch)

    # Calculate overlaps and add them to TensorBoard
    overlap_train = overlap(
        torch.exp(model(samples_train).view(-1)), torch.exp(log_probs_train)
    )
    overlap_test = overlap(
        torch.exp(model(samples_test).view(-1)), torch.exp(log_probs_test)
    )

    writer.add_scalar("train/overlap", overlap_train, epoch)
    # writer.add_scalar('Overlap Test', overlap_test, epoch)
    writer.add_scalar("test/overlap", overlap_test, epoch)

writer.close()
print("Finished Training")

KeyboardInterrupt: 

154625